# Day 20 — Week 3–4 Mini Project: Sales Performance Report
**Estimated time:** 75–90 min
**All 5 datasets in SQLite**

## Learning Objectives
- Apply all SQL skills in a single end-to-end analytical project
- Combine CTEs, window functions, aggregations, and joins in a real reporting flow
- Pull results into pandas for formatting and generate a text summary report

---
*This project mirrors a SAP BW/4HANA or SAC report request: sales performance by region and period, customer LTV, materials with declining trends, and churn risk identification.*

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = "../../data/raw"

# Load all CSVs
inv   = pd.read_csv(f"{DATA_DIR}/materials_inventory.csv")
sales = pd.read_csv(f"{DATA_DIR}/sales_orders.csv")
cc    = pd.read_csv(f"{DATA_DIR}/cost_center_actuals.csv")
hr    = pd.read_csv(f"{DATA_DIR}/hr_headcount.csv")
bw    = pd.read_csv(f"{DATA_DIR}/bw_sales_kpis.csv")

# Normalize column names
for df in [inv, sales, cc, hr, bw]:
    df.columns = [c.strip().upper() for c in df.columns]

# In-memory DuckDB — register pandas DataFrames as tables
con = duckdb.connect()
con.register("MATERIALS",   inv)
con.register("SALES",       sales)
con.register("COST_CENTER", cc)
con.register("HR",          hr)
con.register("BW_SALES",    bw)

def q(sql):
    return con.sql(sql).df()

# Verify
for tbl, df in [("MATERIALS",inv),("SALES",sales),("COST_CENTER",cc),("HR",hr),("BW_SALES",bw)]:
    n = q(f"SELECT COUNT(*) AS n FROM {tbl}").iloc[0,0]
    print(f"  {tbl:15s}: {n:,} rows")


In [ ]:
# -- Section 1: Revenue by region and period --
revenue_by_region = q(
    """
    WITH monthly_regional AS (
        SELECT
            b.CAL_YEAR_MONTH   AS period,
            b.REGION,
            b.VKORG,
            SUM(b.REVENUE)     AS revenue,
            SUM(b.GROSS_MARGIN) AS gross_margin,
            SUM(b.QUANTITY)    AS total_qty,
            COUNT(DISTINCT b.KUNNR) AS n_customers
        FROM BW_SALES b
        GROUP BY b.CAL_YEAR_MONTH, b.REGION, b.VKORG
    )
    SELECT
        period,
        REGION,
        VKORG,
        ROUND(revenue, 2)      AS revenue,
        ROUND(gross_margin, 2) AS gross_margin,
        ROUND(gross_margin * 100.0 / NULLIF(revenue, 0), 2) AS gm_pct,
        total_qty,
        n_customers,
        ROUND(LAG(revenue) OVER (PARTITION BY REGION, VKORG ORDER BY period), 2) AS prev_period_rev,
        ROUND(
            (revenue - LAG(revenue) OVER (PARTITION BY REGION, VKORG ORDER BY period)) * 100.0
            / NULLIF(LAG(revenue) OVER (PARTITION BY REGION, VKORG ORDER BY period), 0)
        , 2) AS mom_pct_change
    FROM monthly_regional
    ORDER BY period DESC, revenue DESC
    """
)
print(f"Revenue by region/period: {len(revenue_by_region):,} rows")
display(revenue_by_region.head(20))

In [ ]:
# -- Section 2: Top customers by lifetime value (LTV) --
customer_ltv = q(
    """
    WITH customer_orders AS (
        SELECT
            KUNNR,
            COUNT(DISTINCT VBELN)   AS total_orders,
            SUM(NETWR)              AS lifetime_revenue,
            MIN(ERDAT)              AS first_order,
            MAX(ERDAT)              AS last_order,
            SUM(CASE WHEN STATUS = 'CANCELLED' THEN 1.0 ELSE 0 END)
                / COUNT(*)          AS cancel_rate,
            ROUND(
                (julianday(MAX(ERDAT)) - julianday(MIN(ERDAT))) / 365.25
            , 2)                    AS customer_tenure_years
        FROM SALES
        GROUP BY KUNNR
    )
    SELECT
        KUNNR,
        total_orders,
        ROUND(lifetime_revenue, 2) AS lifetime_revenue,
        first_order,
        last_order,
        ROUND(cancel_rate * 100, 2) AS cancel_rate_pct,
        customer_tenure_years,
        DENSE_RANK() OVER (ORDER BY lifetime_revenue DESC) AS ltv_rank
    FROM customer_orders
    ORDER BY lifetime_revenue DESC
    LIMIT 25
    """
)
display(customer_ltv)

In [ ]:
# -- Section 3: Materials with declining sales trend --
# Declining = last 3 months revenue < prior 3 months revenue (same period)
material_trend = q(
    """
    WITH monthly_mat AS (
        SELECT
            MATNR,
            SUBSTR(ERDAT, 1, 7) AS ym,
            SUM(NETWR) AS revenue,
            SUM(MENGE) AS qty
        FROM SALES
        WHERE ERDAT IS NOT NULL
        GROUP BY MATNR, ym
    ),
    recent AS (
        SELECT
            MATNR,
            SUM(CASE WHEN ym >= strftime('%Y-%m', date('now', '-3 months')) THEN revenue ELSE 0 END) AS recent_3m,
            SUM(CASE WHEN ym >= strftime('%Y-%m', date('now', '-6 months'))
                      AND ym  < strftime('%Y-%m', date('now', '-3 months')) THEN revenue ELSE 0 END) AS prior_3m
        FROM monthly_mat
        GROUP BY MATNR
    )
    SELECT
        r.MATNR,
        m.MAKTX,
        ROUND(r.recent_3m, 2) AS recent_3m_revenue,
        ROUND(r.prior_3m,  2) AS prior_3m_revenue,
        ROUND((r.recent_3m - r.prior_3m) * 100.0 / NULLIF(r.prior_3m, 0), 2) AS trend_pct,
        CASE WHEN r.recent_3m < r.prior_3m THEN 'Declining' ELSE 'Growing' END AS trend
    FROM recent r
    LEFT JOIN (SELECT DISTINCT MATNR, MAKTX FROM MATERIALS) m ON r.MATNR = m.MATNR
    WHERE r.prior_3m > 0
    ORDER BY trend_pct ASC
    LIMIT 20
    """
)
print("Materials with steepest declining sales trend:")
display(material_trend)

In [ ]:
# -- Section 4: Customers at risk (high cancellation rate) --
at_risk_customers = q(
    """
    WITH customer_stats AS (
        SELECT
            KUNNR,
            COUNT(*)   AS total_orders,
            SUM(CASE WHEN STATUS = 'CANCELLED' THEN 1.0 ELSE 0 END) AS cancelled,
            SUM(CASE WHEN STATUS = 'OPEN' THEN NETWR ELSE 0 END)    AS open_order_value,
            MAX(ERDAT) AS last_order_date
        FROM SALES
        GROUP BY KUNNR
        HAVING COUNT(*) >= 5
    )
    SELECT
        KUNNR,
        total_orders,
        ROUND(cancelled / total_orders * 100, 2) AS cancel_rate_pct,
        ROUND(open_order_value, 2) AS open_order_value,
        last_order_date,
        ROUND(julianday('now') - julianday(last_order_date), 0) AS days_since_last_order,
        CASE
            WHEN cancelled / total_orders > 0.30 THEN 'HIGH RISK'
            WHEN cancelled / total_orders > 0.15 THEN 'MEDIUM RISK'
            ELSE 'LOW RISK'
        END AS risk_tier
    FROM customer_stats
    WHERE cancelled / total_orders > 0.15
    ORDER BY cancel_rate_pct DESC
    LIMIT 20
    """
)
print("At-risk customers (cancellation rate > 15%, min 5 orders):")
display(at_risk_customers)

In [ ]:
# -- Formatted text summary report --
total_revenue    = q("SELECT ROUND(SUM(NETWR), 2) AS r FROM SALES").iloc[0,0]
total_customers  = q("SELECT COUNT(DISTINCT KUNNR) AS n FROM SALES").iloc[0,0]
top_region       = revenue_by_region.groupby("REGION")["REVENUE"].sum().idxmax() if "REGION" in revenue_by_region.columns else "N/A"
n_declining      = (material_trend["TREND"] == "Declining").sum()
n_high_risk_custs = (at_risk_customers["RISK_TIER"] == "HIGH RISK").sum()

report = f"""
=======================================================
  SAP ANALYTICS BOOTCAMP — SALES PERFORMANCE REPORT
=======================================================

OVERALL PERFORMANCE
  Total revenue (all time):  ${total_revenue:,.2f}
  Distinct customers:        {total_customers:,}
  Top region by revenue:     {top_region}

CUSTOMER HEALTH
  Customers at HIGH risk:    {n_high_risk_custs}
  (cancel rate > 30%, min 5 orders)

MATERIAL TRENDS
  Materials with declining
  3-month revenue trend:     {n_declining}

TOP 5 CUSTOMERS BY LTV
{customer_ltv[['KUNNR','lifetime_revenue','ltv_rank']].head(5).to_string(index=False)}

=======================================================
"""
print(report)

with open("day20_sales_report.txt", "w") as f:
    f.write(report)
print("Saved: day20_sales_report.txt")

In [ ]:
# -- Export all sections to CSV --
revenue_by_region.to_csv("day20_revenue_by_region.csv", index=False)
customer_ltv.to_csv("day20_customer_ltv.csv", index=False)
material_trend.to_csv("day20_material_trends.csv", index=False)
at_risk_customers.to_csv("day20_at_risk_customers.csv", index=False)

print("All outputs saved:")
import os
for f in ["day20_revenue_by_region.csv","day20_customer_ltv.csv","day20_material_trends.csv","day20_at_risk_customers.csv","day20_sales_report.txt"]:
    if os.path.exists(f):
        print(f"  {f}  ({os.path.getsize(f):,} bytes)")

---
## Self-Grade Rubric

| Dimension | 1 — Needs Work | 2 — Functional | 3 — Solid | 4 — Strong |
|-----------|---------------|----------------|-----------|------------|
| **Revenue by region** | Wrong aggregation | Correct totals | + MoM change | + formatted + trend direction |
| **Customer LTV** | Partial fields | All fields correct | + cancel rate | + LTV rank + tenure |
| **Declining materials** | Hard-coded periods | Correct 3m/6m logic | Parameterized periods | + visualization |
| **At-risk customers** | Single condition | Multi-condition risk | + days since last order | + action recommendation |
| **Text report** | Print only | f-string formatted | + all sections | + saved to file |
| **Code quality** | Working SQL | Clean CTEs | Reusable helper functions | + type hints + error handling |

**Target:** Score 3+ before completing the bootcamp.

---
## Stretch Challenges

1. **Cohort analysis**: group customers by their first order month and track retention over time
2. **Pareto / ABC**: identify the 20% of customers driving 80% of revenue (Pareto principle in SQL)
3. **Funnel analysis**: orders by status → what % convert from OPEN → COMPLETED vs CANCELLED?
4. **Python automation**: wrap the entire report in a function `generate_sales_report(as_of_date)` that accepts a date and regenerates all sections point-in-time